# Partie 3 - Spark SQL et DataFrames
Refaire et enrichir l'analyse de la Partie 2 avec l'API structuree (schemas explicites, SQL,
jointure, UDF, Parquet partitionne, plans Catalyst).

In [1]:
import os, sys, json, statistics
sys.path.insert(0, "/home/sahlpay/jobs"); sys.path.insert(0, "../jobs")
from sahlpay_common import get_spark, Paths, LOG_DIR, TX_SCHEMA, MERCHANT_SCHEMA, merchants_path, timer
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

spark = get_spark("SahlPay-P3-SQL", cores_max=4)
sc = spark.sparkContext
timings = {}
os.makedirs(LOG_DIR, exist_ok=True)

26/11/18 21:46:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/18 21:46:51 INFO StandaloneAppClient$ClientEndpoint: Connecting to master spark://spark-master:7077...
26/11/18 21:46:51 INFO StandaloneAppClient$ClientEndpoint: Executor added: app-20261118214651-0011/0 on worker-20261118205754-spark-worker-1-43731 (spark-worker-1:43731) with 1 core(s)
26/11/18 21:46:51 INFO StandaloneAppClient$ClientEndpoint: Executor added: app-20261118214651-0011/1 on worker-20261118205754-spark-worker-1-43731 (spark-worker-1:43731) with 1 core(s)
26/11/18 21:46:51 INFO StandaloneAppClient$ClientEndpoint: Executor added: app-20261118214651-0011/2 on worker-20261118205754-spark-worker-2-45013 (spark-worker-2:45013) with 1 core(s)
26/11/18 21:46:51 INFO StandaloneAppClient$ClientEndpoint: Executor added: app-20261118214651-0011/3 on worker-20261118205754-spark-worker-2-45013 (spark-worker-2:45013) with 1 core(s)
26/11/18

## 1. Lecture avec schema explicite (`StructType`, sans inference)

In [2]:
history_file = f"{Paths.bronze_history}/transactions_history.jsonl"
tx = (spark.read.schema(TX_SCHEMA)
      .option("mode", "PERMISSIVE")
      .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ssX")
      .json(history_file))
merchants = spark.read.schema(MERCHANT_SCHEMA).json(merchants_path())
tx.printSchema()
merchants.printSchema()
with timer("p3_count_events", timings):
    print("evenements :", tx.count(), "| merchants :", merchants.count())

root
 |-- tx_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- payer_wallet: string (nullable = true)
 |-- payee_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount_mad: double (nullable = true)
 |-- event_time: timestamp (nullable = true)

root
 |-- merchant_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)

evenements : 50000 | merchants : 300
[timer] p3_count_events: 11.23 s


## 2. Montant total (et moyen) par ville : API DataFrame puis SQL
Meme regle qu'en Partie 2 : transactions `COMPLETED`, dedoublonnees par `tx_id`.

In [3]:
completed = tx.filter(F.col("status") == "COMPLETED").dropDuplicates(["tx_id"])

with timer("p3_total_by_city_df", timings):
    by_city_df = (completed.groupBy("city")
                  .agg(F.round(F.sum("amount_mad"), 2).alias("montant_total_mad"),
                       F.count("*").alias("nb_tx"),
                       F.round(F.avg("amount_mad"), 2).alias("montant_moyen_mad"))
                  .orderBy(F.desc("montant_total_mad")))
    by_city_df.show(truncate=False)

+----------+-----------------+-----+-----------------+
|city      |montant_total_mad|nb_tx|montant_moyen_mad|
+----------+-----------------+-----+-----------------+
|Casablanca|1304409.66       |4410 |295.78           |
|Rabat     |588172.85        |2096 |280.62           |
|Marrakech |586394.46        |1813 |323.44           |
|Fes       |415827.47        |1549 |268.45           |
|Tanger    |405721.56        |1437 |282.34           |
|Agadir    |354117.18        |1117 |317.03           |
|Tetouan   |255874.61        |748  |342.08           |
|Kenitra   |248381.51        |838  |296.4            |
|Meknes    |190841.97        |814  |234.45           |
|Oujda     |170806.81        |713  |239.56           |
+----------+-----------------+-----+-----------------+

[timer] p3_total_by_city_df: 6.92 s


In [4]:
tx.createOrReplaceTempView("transactions")
merchants.createOrReplaceTempView("merchants")

with timer("p3_total_by_city_sql", timings):
    by_city_sql = spark.sql("""
        WITH completed AS (
            SELECT tx_id, first(city) AS city, first(amount_mad) AS amount_mad
            FROM transactions
            WHERE status = 'COMPLETED'
            GROUP BY tx_id
        )
        SELECT city,
               round(sum(amount_mad), 2) AS montant_total_mad,
               count(*)                  AS nb_tx,
               round(avg(amount_mad), 2) AS montant_moyen_mad
        FROM completed
        GROUP BY city
        ORDER BY montant_total_mad DESC
    """)
    by_city_sql.show(truncate=False)

# Les deux resultats sont identiques
assert sorted(map(tuple, by_city_df.collect())) == sorted(map(tuple, by_city_sql.collect()))
print("DataFrame et SQL donnent le meme resultat")

+----------+-----------------+-----+-----------------+
|city      |montant_total_mad|nb_tx|montant_moyen_mad|
+----------+-----------------+-----+-----------------+
|Casablanca|1304409.66       |4410 |295.78           |
|Rabat     |588172.85        |2096 |280.62           |
|Marrakech |586394.46        |1813 |323.44           |
|Fes       |415827.47        |1549 |268.45           |
|Tanger    |405721.56        |1437 |282.34           |
|Agadir    |354117.18        |1117 |317.03           |
|Tetouan   |255874.61        |748  |342.08           |
|Kenitra   |248381.51        |838  |296.4            |
|Meknes    |190841.97        |814  |234.45           |
|Oujda     |170806.81        |713  |239.56           |
+----------+-----------------+-----+-----------------+

[timer] p3_total_by_city_sql: 2.49 s
DataFrame et SQL donnent le meme resultat


## 3. Jointure P2M x merchants : montant encaisse par categorie et par ville

In [5]:
p2m = completed.filter(F.col("type") == "P2M")
joined = p2m.join(merchants, p2m.payee_id == merchants.merchant_id, "inner")

with timer("p3_join_category_city", timings):
    by_cat_city = (joined.groupBy(merchants.category, merchants.city)
                   .agg(F.round(F.sum("amount_mad"), 2).alias("montant_total_mad"),
                        F.count("*").alias("nb_paiements"))
                   .orderBy(F.desc("montant_total_mad")))
    by_cat_city.show(20, truncate=False)

(joined.groupBy(merchants.category)
       .agg(F.round(F.sum("amount_mad"), 2).alias("montant_total_mad"), F.count("*").alias("nb"))
       .orderBy(F.desc("montant_total_mad"))
       .show(truncate=False))

+------------+----------+-----------------+------------+
|category    |city      |montant_total_mad|nb_paiements|
+------------+----------+-----------------+------------+
|carburant   |Casablanca|166333.87        |483         |
|restauration|Casablanca|135916.62        |934         |
|e-commerce  |Agadir    |134997.87        |203         |
|e-commerce  |Tetouan   |114771.78        |159         |
|e-commerce  |Rabat     |111532.25        |174         |
|epicerie    |Casablanca|104096.6         |995         |
|restauration|Marrakech |91707.03         |644         |
|e-commerce  |Casablanca|88546.02         |120         |
|habillement |Marrakech |72083.79         |165         |
|epicerie    |Fes       |70971.47         |639         |
|habillement |Casablanca|68535.66         |137         |
|carburant   |Rabat     |64679.04         |207         |
|e-commerce  |Oujda     |58806.51         |95          |
|e-commerce  |Kenitra   |58698.98         |77          |
|telecom     |Casablanca|54640.

## 4. UDF `amount_band` vs `when/otherwise`

In [6]:
def amount_band(amount_mad):
    if amount_mad is None:
        return None
    if amount_mad < 50:
        return "micro"
    if amount_mad <= 2000:
        return "standard"
    return "eleve"

amount_band_udf = F.udf(amount_band, StringType())
spark.udf.register("amount_band", amount_band, StringType())   # utilisable aussi en SQL

def amount_band_native(col):
    return (F.when(col < 50, "micro")
             .when(col <= 2000, "standard")
             .otherwise("eleve"))

def bench(df_fn, label, runs=3):
    times = []
    import time
    for _ in range(runs):
        t0 = time.perf_counter()
        df_fn().groupBy("amount_band").count().collect()
        times.append(time.perf_counter() - t0)
    timings[label] = round(statistics.median(times), 3)
    return timings[label]

t_udf = bench(lambda: tx.withColumn("amount_band", amount_band_udf("amount_mad")), "p3_band_udf")
t_nat = bench(lambda: tx.withColumn("amount_band", amount_band_native(F.col("amount_mad"))), "p3_band_when")
print(f"UDF Python : {t_udf:.2f} s | when/otherwise : {t_nat:.2f} s (mediane de 3 executions)")

tx.withColumn("amount_band", amount_band_native(F.col("amount_mad"))).groupBy("amount_band").count().show()
spark.sql("SELECT amount_band(amount_mad) AS band, count(*) AS n FROM transactions GROUP BY 1").show()

print("--- plan physique avec UDF (noter BatchEvalPython) ---")
tx.withColumn("amount_band", amount_band_udf("amount_mad")).explain()
print("--- plan physique avec when/otherwise (CASE WHEN natif, codegen) ---")
tx.withColumn("amount_band", amount_band_native(F.col("amount_mad"))).explain()

UDF Python : 2.01 s | when/otherwise : 0.83 s (mediane de 3 executions)
+-----------+-----+
|amount_band|count|
+-----------+-----+
|      eleve|  809|
|      micro| 7059|
|   standard|42132|
+-----------+-----+

+--------+-----+
|    band|    n|
+--------+-----+
|   eleve|  809|
|   micro| 7059|
|standard|42132|
+--------+-----+

--- plan physique avec UDF (noter BatchEvalPython) ---
== Physical Plan ==
*(1) Project [tx_id#0, type#1, payer_wallet#2, payee_id#3, device_id#4, city#5, channel#6, status#7, amount_mad#8, event_time#9, pythonUDF0#838 AS amount_band#826]
+- BatchEvalPython [amount_band(amount_mad#8)#825], [pythonUDF0#838]
   +- FileScan json [tx_id#0,type#1,payer_wallet#2,payee_id#3,device_id#4,city#5,channel#6,status#7,amount_mad#8,event_time#9] Batched: false, DataFilters: [], Format: JSON, Location: InMemoryFileIndex(1 paths)[s3a://lake/bronze/history/transactions_history.jsonl], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<tx_id:string,type:string,payer_wa

**Pourquoi `when/otherwise` est plus performant** : une UDF Python est une boite noire pour
Catalyst. Chaque ligne doit etre serialisee depuis la JVM vers un processus Python (operateur
`BatchEvalPython`, via Arrow/pickle), evaluee en Python puis renvoyee. `when/otherwise` est
traduit en `CASE WHEN` natif : il reste dans la JVM, beneficie du *whole-stage code generation*
de Tungsten (format binaire UnsafeRow, pas de serialisation) et peut etre optimise par Catalyst
(constant folding, predicate pushdown...).

## 5. Ecriture de la couche Silver (Parquet partitionne par `city`)
Nettoyage : schema type, suppression des lignes invalides, dedoublonnage `(tx_id, status)`,
ajout de `amount_band`, `date`, et des attributs commercant pour les paiements P2M.

In [7]:
silver = (tx
          .filter(F.col("tx_id").isNotNull() & F.col("event_time").isNotNull() & (F.col("amount_mad") > 0))
          .dropDuplicates(["tx_id", "status"])
          .withColumn("amount_band", amount_band_native(F.col("amount_mad")))
          .withColumn("date", F.to_date("event_time"))
          .join(F.broadcast(merchants.select(F.col("merchant_id").alias("payee_id"),
                                             F.col("name").alias("merchant_name"),
                                             F.col("category").alias("merchant_category"))),
                on="payee_id", how="left"))

with timer("p3_write_silver", timings):
    (silver.repartition("city")
           .write.mode("overwrite")
           .partitionBy("city")
           .parquet(Paths.silver_history))

silver_back = spark.read.parquet(Paths.silver_history)
print("lignes Silver :", silver_back.count())
silver_back.groupBy("city").count().orderBy("city").show()
# Lecture d'une seule partition : partition pruning (PartitionFilters dans le plan)
silver_back.filter(F.col("city") == "Rabat").groupBy("status").count().explain()

[timer] p3_write_silver: 9.00 s
lignes Silver : 49065
+----------+-----+
|      city|count|
+----------+-----+
|    Agadir| 3497|
|Casablanca|13956|
|       Fes| 4886|
|   Kenitra| 2625|
| Marrakech| 5732|
|    Meknes| 2582|
|     Oujda| 2327|
|     Rabat| 6546|
|    Tanger| 4574|
|   Tetouan| 2340|
+----------+-----+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[status#1026], functions=[count(1)])
   +- Exchange hashpartitioning(status#1026, 8), ENSURE_REQUIREMENTS, [plan_id=2056]
      +- HashAggregate(keys=[status#1026], functions=[partial_count(1)])
         +- Project [status#1026]
            +- FileScan parquet [status#1026,city#1033] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[s3a://lake/silver/transactions_history], PartitionFilters: [isnotnull(city#1033), (city#1033 = Rabat)], PushedFilters: [], ReadSchema: struct<status:string>




## 6. `explain(True)` sur la jointure : plans logique, optimise (Catalyst) et physique

In [8]:
join_query = (p2m.join(merchants, p2m.payee_id == merchants.merchant_id)
                 .groupBy(merchants.category, merchants.city)
                 .agg(F.sum("amount_mad").alias("montant_total_mad")))
join_query.explain(True)

with open(os.path.join(LOG_DIR, "explain_join_broadcast.txt"), "w") as f:
    f.write(join_query._jdf.queryExecution().toString())

== Parsed Logical Plan ==
'Aggregate [category#23, city#22], [category#23, city#22, sum('amount_mad) AS montant_total_mad#1166]
+- Join Inner, (payee_id#3 = merchant_id#20)
   :- Filter (type#1 = P2M)
   :  +- Deduplicate [tx_id#0]
   :     +- Filter (status#7 = COMPLETED)
   :        +- Relation [tx_id#0,type#1,payer_wallet#2,payee_id#3,device_id#4,city#5,channel#6,status#7,amount_mad#8,event_time#9] json
   +- Relation [merchant_id#20,name#21,city#22,category#23,latitude#24,longitude#25] json

== Analyzed Logical Plan ==
category: string, city: string, montant_total_mad: double
Aggregate [category#23, city#22], [category#23, city#22, sum(amount_mad#8) AS montant_total_mad#1166]
+- Join Inner, (payee_id#3 = merchant_id#20)
   :- Filter (type#1 = P2M)
   :  +- Deduplicate [tx_id#0]
   :     +- Filter (status#7 = COMPLETED)
   :        +- Relation [tx_id#0,type#1,payer_wallet#2,payee_id#3,device_id#4,city#5,channel#6,status#7,amount_mad#8,event_time#9] json
   +- Relation [merchant_id#2

Pour comparaison : on desactive le broadcast automatique, Spark choisit alors un **SortMergeJoin**
(shuffle des deux cotes + tri). Avec `merchants` (300 lignes, quelques Ko, sous le seuil
`spark.sql.autoBroadcastJoinThreshold` = 10 Mo), Catalyst choisit par defaut un
**BroadcastHashJoin** : la petite table est envoyee a chaque executor, la grande n'est pas shufflee.

In [9]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
with timer("p3_join_sortmerge", timings):
    join_query_smj = (p2m.join(merchants, p2m.payee_id == merchants.merchant_id)
                         .groupBy(merchants.category, merchants.city)
                         .agg(F.sum("amount_mad").alias("montant_total_mad")))
    join_query_smj.collect()
join_query_smj.explain()
with open(os.path.join(LOG_DIR, "explain_join_sortmerge.txt"), "w") as f:
    f.write(join_query_smj._jdf.queryExecution().toString())
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
with timer("p3_join_broadcast", timings):
    join_query.collect()

[timer] p3_join_sortmerge: 1.97 s
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   *(8) HashAggregate(keys=[category#23, city#22], functions=[sum(amount_mad#1270)])
   +- AQEShuffleRead coalesced
      +- ShuffleQueryStage 3
         +- Exchange hashpartitioning(category#23, city#22, 8), ENSURE_REQUIREMENTS, [plan_id=2447]
            +- *(7) HashAggregate(keys=[category#23, city#22], functions=[partial_sum(amount_mad#1270)])
               +- *(7) Project [amount_mad#1270, city#22, category#23]
                  +- *(7) SortMergeJoin [payee_id#1260], [merchant_id#20], Inner
                     :- *(5) Sort [payee_id#1260 ASC NULLS FIRST], false, 0
                     :  +- AQEShuffleRead coalesced
                     :     +- ShuffleQueryStage 2
                     :        +- Exchange hashpartitioning(payee_id#1260, 8), ENSURE_REQUIREMENTS, [plan_id=2371]
                     :           +- *(4) Project [payee_id#1260, amount_mad#1270]
               

## 7. Comparaison RDD / DataFrame / Dataset (temps mesures)

In [10]:
try:
    with open(os.path.join(LOG_DIR, "timings_part2_rdd.json")) as f:
        t2 = json.load(f)
except FileNotFoundError:
    t2 = {}
rows = [
    ("Lecture + comptage",        t2.get("p2_count_events"),   timings.get("p3_count_events")),
    ("Montant total par ville",   t2.get("p2_total_by_city"),  timings.get("p3_total_by_city_df")),
    ("Total par ville (SQL)",     None,                        timings.get("p3_total_by_city_sql")),
    ("Total + moyenne par ville", t2.get("p2_avg_by_city"),    timings.get("p3_total_by_city_df")),
    ("DAG complet sans cache",    t2.get("p2_dag_job_nocache"), timings.get("p3_total_by_city_df")),
]
print(f"{'Traitement':<28}{'RDD (s)':>10}{'DataFrame (s)':>15}")
for r in rows:
    print(f"{r[0]:<28}{(r[1] if r[1] is not None else float('nan')):>10.2f}{(r[2] if r[2] is not None else float('nan')):>15.2f}")
with open(os.path.join(LOG_DIR, "timings_part3_df.json"), "w") as f:
    json.dump(timings, f, indent=2)
print(timings)

Traitement                     RDD (s)  DataFrame (s)
Lecture + comptage               11.59          11.23
Montant total par ville           0.47           6.92
Total par ville (SQL)              nan           2.49
Total + moyenne par ville         0.42           6.92
DAG complet sans cache            0.97           6.92
{'p3_count_events': 11.226, 'p3_total_by_city_df': 6.92, 'p3_total_by_city_sql': 2.491, 'p3_join_category_city': 2.963, 'p3_band_udf': 2.012, 'p3_band_when': 0.83, 'p3_write_silver': 8.998, 'p3_join_sortmerge': 1.969, 'p3_join_broadcast': 1.47}


**A rediger dans le rapport**
* **RDD** : collection distribuee d'objets Python opaques ; aucune connaissance du schema, pas
  d'optimisation automatique ; chaque `lambda` s'execute dans un worker Python (serialisation
  JVM <-> Python). Controle fin mais code plus long.
* **DataFrame** : donnees tabulaires avec schema ; les requetes (API ou SQL) produisent le meme
  plan logique optimise par **Catalyst** (predicate pushdown, projection pruning, choix du join) et
  execute par **Tungsten** (memoire hors tas, format binaire, code genere). En PySpark, tout le
  calcul reste dans la JVM tant qu'on n'utilise pas d'UDF Python.
* **Dataset** : API typee (Scala/Java) qui combine typage statique a la compilation et Catalyst ;
  n'existe pas en Python (un DataFrame est un `Dataset[Row]`). Les lambdas typees d'un Dataset
  (`map(x => ...)`) sont aussi opaques pour Catalyst.
* Nos mesures : sur 11 Mo, le RDD (deja en cache) sort devant ; le DataFrame relit le JSON, fait deux shuffles et paie le cout fixe de Catalyst. Son avantage apparait quand le calcul domine (when/otherwise 2,4 fois plus rapide que l'UDF) et sur de gros volumes (moins de
  serialisation, lecture JSON vectorisee, agregation codegen), et SQL = DataFrame (meme plan).

In [12]:
spark.stop()